# Kaggle Execution Notebook For TCRB Finetuning

This notebook is designed to run inside a Kaggle notebook runtime.

It does four things in one place:

1. Clones the current repository branch into `/kaggle/working`
2. Installs the dependencies needed for the research pipeline
3. Runs the repo commands for data prep, mining, training, and validation
4. Stages and packages all outputs so they can be downloaded locally and committed back to the repository

In [1]:
!nvidia-smi

Fri Apr 24 06:01:36 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.105.08             Driver Version: 580.105.08     CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   41C    P8              9W /   70W |       0MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

## 1. Configure Kaggle Paths And Runtime Variables

These variables centralize the clone target, branch, output staging paths, and command toggles so the notebook can be rerun without rewriting shell commands.

In [2]:
from __future__ import annotations

import json
import shutil
import subprocess
from datetime import UTC, datetime
from pathlib import Path

REPO_URL = "https://github.com/aaliyan1230/tool-calling-reliability-benchmark.git"
BRANCH_NAME = "feat/llm"
REPO_DIR = Path("/kaggle/working/tool-calling-reliability-benchmark")
WORKING_DIR = Path("/kaggle/working")
INPUT_DIR = Path("/kaggle/input")
EXPORT_ROOT = WORKING_DIR / "tcrb_kaggle_exports"
ARTIFACT_STAGE_DIR = EXPORT_ROOT / "staged_artifacts"
PATCH_EXPORT_DIR = EXPORT_ROOT / "git_bundle"
RUN_STAMP = datetime.now(UTC).strftime("%Y%m%d-%H%M%S")
SFT_RECIPE = "configs/research/sft_toolace_qwen25_3b.json"
MASKED_SFT_RECIPE = "configs/research/sft_toolace_hermes_mask20_qwen25_3b.json"
DPO_RECIPE = "configs/research/dpo_failure_qwen25_3b.json"
EVAL_CASES_JSON = "workloads/eval_cases/customer_support_eval_cases.json"
BENCHMARK_POLICY = "naive_retry"
HF_RESULT_JSON = "runs/hf-run/result.json"
FAILURE_PAIR_JSONL = "outputs/research/qwen25-3b-failure-pairs/dpo_train.jsonl"
RUN_SFT = False
RUN_MASKED_SFT = False
RUN_DPO = False
RUN_HF_BENCHMARK = False
RUN_TESTS = True
PUSH_TO_REMOTE = False
PUSH_REMOTE_NAME = "origin"
PUSH_BRANCH_NAME = BRANCH_NAME

EXPORT_ROOT.mkdir(parents=True, exist_ok=True)
ARTIFACT_STAGE_DIR.mkdir(parents=True, exist_ok=True)
PATCH_EXPORT_DIR.mkdir(parents=True, exist_ok=True)

config_summary = {
    "repo_url": REPO_URL,
    "branch_name": BRANCH_NAME,
    "repo_dir": str(REPO_DIR),
    "working_dir": str(WORKING_DIR),
    "input_dir": str(INPUT_DIR),
    "export_root": str(EXPORT_ROOT),
    "run_sft": RUN_SFT,
    "run_masked_sft": RUN_MASKED_SFT,
    "run_dpo": RUN_DPO,
    "run_hf_benchmark": RUN_HF_BENCHMARK,
    "run_tests": RUN_TESTS,
    "push_to_remote": PUSH_TO_REMOTE,
}
print(json.dumps(config_summary, indent=2))
print("Working dir exists:", WORKING_DIR.exists())
print("Input dir exists:", INPUT_DIR.exists())

{
  "repo_url": "https://github.com/aaliyan1230/tool-calling-reliability-benchmark.git",
  "branch_name": "feat/llm",
  "repo_dir": "/kaggle/working/tool-calling-reliability-benchmark",
  "working_dir": "/kaggle/working",
  "input_dir": "/kaggle/input",
  "export_root": "/kaggle/working/tcrb_kaggle_exports",
  "run_sft": false,
  "run_masked_sft": false,
  "run_dpo": false,
  "run_hf_benchmark": false,
  "run_tests": true,
  "push_to_remote": false
}
Working dir exists: True
Input dir exists: True


## 2. Clone The Repository And Checkout The Target Branch

This cell safely reuses the repo if it already exists in the Kaggle session, otherwise it clones the requested branch from GitHub.

In [3]:
%%bash
set -euo pipefail

REPO_DIR="/kaggle/working/tool-calling-reliability-benchmark"
REPO_URL="https://github.com/aaliyan1230/tool-calling-reliability-benchmark.git"
BRANCH_NAME="feat/llm"

if [[ -d "$REPO_DIR/.git" ]]; then
  echo "Repository already exists at $REPO_DIR"
  cd "$REPO_DIR"
  git remote set-url origin "$REPO_URL"
  git fetch origin --prune
else
  rm -rf "$REPO_DIR"
  git clone --branch "$BRANCH_NAME" "$REPO_URL" "$REPO_DIR"
  cd "$REPO_DIR"
fi

git fetch origin "$BRANCH_NAME"
git checkout "$BRANCH_NAME"
git reset --hard "origin/$BRANCH_NAME"
echo "Clone and checkout complete."

Your branch is up to date with 'origin/feat/llm'.
HEAD is now at a109a81 feat: add mining functionality for benchmark failure pairs and corresponding CLI commands
Clone and checkout complete.


Cloning into '/kaggle/working/tool-calling-reliability-benchmark'...
From https://github.com/aaliyan1230/tool-calling-reliability-benchmark
 * branch            feat/llm   -> FETCH_HEAD
Already on 'feat/llm'


## 3. Inspect Repository Contents And Git State

Run this before any training or export step to confirm the correct branch, commit, and working directory.

In [4]:
%%bash
set -euo pipefail
cd /kaggle/working/tool-calling-reliability-benchmark
pwd
ls
printf '\nCurrent branch:\n'
git branch --show-current
printf '\nGit status:\n'
git status --short --branch
printf '\nLatest commit:\n'
git log -1 --oneline
printf '\nAvailable research configs:\n'
ls configs/research

/kaggle/working/tool-calling-reliability-benchmark
configs
docs
examples
notebooks
pyproject.toml
README.md
scripts
src
tests
uv.lock
workloads

Current branch:
feat/llm

Git status:
## feat/llm...origin/feat/llm

Latest commit:
a109a81 feat: add mining functionality for benchmark failure pairs and corresponding CLI commands

Available research configs:
dpo_failure_qwen25_3b.json
dpo_toolpreference_qwen25_3b.json
sft_toolace_hermes_mask20_qwen25_3b.json
sft_toolace_qwen25_3b.json


## 4. Install Project Dependencies In The Kaggle Environment

This notebook uses the repo's optional `research` extras so the same commands work in Kaggle that we added locally.

In [ ]:
%%bash
set -euo pipefail
cd /kaggle/working/tool-calling-reliability-benchmark
python --version
pip --version
python -m pip install --upgrade pip setuptools wheel
python -m pip install uv
uv sync --extra dev --extra research
uv pip install --python .venv/bin/python wrapt
uv run python -c "import accelerate, datasets, peft, transformers, trl, wrapt; print('research dependencies ready')"

Python 3.12.12
pip 26.0.1 from /usr/local/lib/python3.12/dist-packages/pip (python 3.12)
wrapt 2.1.2


Resolved 90 packages in 1ms
Checked 88 packages in 1ms
Resolved 1 package in 251ms
Prepared 1 package in 41ms
         If the cache and target directories are on different filesystems, hardlinking may not be supported.
         If this is intentional, set `export UV_LINK_MODE=copy` or use `--link-mode=copy` to suppress this warning.
Installed 1 package in 4ms
 + wrapt==2.1.2
/kaggle/working/tool-calling-reliability-benchmark/.venv/bin/python3: No module named pip


## 5. Run Project Commands From Kaggle Shell

The cell below is parameterized from the variables in Cell 3. Leave the training toggles set to `False` until the Kaggle runtime is attached to a GPU.

In [12]:
command_lines = [
    "set -euo pipefail",
    "cd /kaggle/working/tool-calling-reliability-benchmark",
    "source .venv/bin/activate",
    "export PYTHONPATH=$PWD/src${PYTHONPATH:+:$PYTHONPATH}",
    "export MPLBACKEND=Agg",
    f"uv run tcrb prepare-sft-data --recipe-config {SFT_RECIPE}",
    f"uv run tcrb prepare-sft-data --recipe-config {MASKED_SFT_RECIPE}",
]

if RUN_TESTS:
    command_lines.append("uv run pytest -q")

if RUN_HF_BENCHMARK:
    command_lines.append(
        "uv run python scripts/run_northstar_hf.py "
        "--base-planner-config configs/planners/policy_native.json "
        "--comparison-planner-config configs/planners/hf_qwen2_5_3b_base.json "
        "--run-study-gate --run-summarize"
    )
    command_lines.append(
        f"uv run tcrb mine-benchmark-failure-pairs --result-json {HF_RESULT_JSON} "
        f"--eval-cases-json {EVAL_CASES_JSON} --policy {BENCHMARK_POLICY} "
        f"--output-jsonl {FAILURE_PAIR_JSONL}"
    )

if RUN_SFT:
    command_lines.append(f"uv run tcrb train-sft --recipe-config {SFT_RECIPE}")

if RUN_MASKED_SFT:
    command_lines.append(f"uv run tcrb train-sft --recipe-config {MASKED_SFT_RECIPE}")

if RUN_DPO:
    command_lines.append(
        f"uv run tcrb train-dpo --recipe-config {DPO_RECIPE} "
        f"--dataset-jsonl {FAILURE_PAIR_JSONL}"
    )

script_text = "\n".join(command_lines)
print(script_text)
subprocess.run(["bash", "-lc", script_text], check=True, cwd=str(REPO_DIR))

set -euo pipefail
cd /kaggle/working/tool-calling-reliability-benchmark
source .venv/bin/activate
export PYTHONPATH=$PWD/src${PYTHONPATH:+:$PYTHONPATH}
export MPLBACKEND=Agg
uv run tcrb prepare-sft-data --recipe-config configs/research/sft_toolace_qwen25_3b.json
uv run tcrb prepare-sft-data --recipe-config configs/research/sft_toolace_hermes_mask20_qwen25_3b.json
uv run pytest -q


Prepared SFT rows: 11300
Wrote SFT JSONL: outputs/research/qwen25-3b-sft-toolace/sft_train.jsonl


Prepared SFT rows: 13193
Wrote SFT JSONL: outputs/research/qwen25-3b-sft-toolace-hermes-mask20/sft_train.jsonl
...................................                                      [100%]
35 passed in 4.18s


CompletedProcess(args=['bash', '-lc', 'set -euo pipefail\ncd /kaggle/working/tool-calling-reliability-benchmark\nsource .venv/bin/activate\nexport PYTHONPATH=$PWD/src${PYTHONPATH:+:$PYTHONPATH}\nexport MPLBACKEND=Agg\nuv run tcrb prepare-sft-data --recipe-config configs/research/sft_toolace_qwen25_3b.json\nuv run tcrb prepare-sft-data --recipe-config configs/research/sft_toolace_hermes_mask20_qwen25_3b.json\nuv run pytest -q'], returncode=0)

## 6. Save Generated Results To A Staging Directory

This copies the outputs most likely to matter for local commit workflow: runs, research outputs, logs, and the notebook itself.

In [13]:
def copy_if_exists(source: Path, destination: Path) -> None:
    if not source.exists():
        print(f"Skip missing path: {source}")
        return
    if source.is_dir():
        shutil.copytree(source, destination, dirs_exist_ok=True)
    else:
        destination.parent.mkdir(parents=True, exist_ok=True)
        shutil.copy2(source, destination)
    print(f"Staged: {source} -> {destination}")

stage_root = ARTIFACT_STAGE_DIR / RUN_STAMP
stage_root.mkdir(parents=True, exist_ok=True)

paths_to_stage = [
    REPO_DIR / "runs",
    REPO_DIR / "outputs" / "research",
    REPO_DIR / "configs" / "research",
    REPO_DIR / "README.md",
    REPO_DIR / "pyproject.toml",
    REPO_DIR / "notebooks" / "kaggle_execution_ft_pipeline.ipynb",
]

for source_path in paths_to_stage:
    target_path = stage_root / source_path.relative_to(REPO_DIR)
    copy_if_exists(source_path, target_path)

manifest_path = stage_root / "export_manifest.json"
manifest_payload = {
    "run_stamp": RUN_STAMP,
    "repo_dir": str(REPO_DIR),
    "staged_paths": [str(path.relative_to(stage_root)) for path in stage_root.rglob("*")],
}
manifest_path.write_text(json.dumps(manifest_payload, indent=2), encoding="utf-8")
print("Stage root:", stage_root)
print("Manifest:", manifest_path)

Skip missing path: /kaggle/working/tool-calling-reliability-benchmark/runs
Staged: /kaggle/working/tool-calling-reliability-benchmark/outputs/research -> /kaggle/working/tcrb_kaggle_exports/staged_artifacts/20260424-055301/outputs/research
Staged: /kaggle/working/tool-calling-reliability-benchmark/configs/research -> /kaggle/working/tcrb_kaggle_exports/staged_artifacts/20260424-055301/configs/research
Staged: /kaggle/working/tool-calling-reliability-benchmark/README.md -> /kaggle/working/tcrb_kaggle_exports/staged_artifacts/20260424-055301/README.md
Staged: /kaggle/working/tool-calling-reliability-benchmark/pyproject.toml -> /kaggle/working/tcrb_kaggle_exports/staged_artifacts/20260424-055301/pyproject.toml
Skip missing path: /kaggle/working/tool-calling-reliability-benchmark/notebooks/kaggle_execution_ft_pipeline.ipynb
Stage root: /kaggle/working/tcrb_kaggle_exports/staged_artifacts/20260424-055301
Manifest: /kaggle/working/tcrb_kaggle_exports/staged_artifacts/20260424-055301/export_m

## 7. Package Outputs For Download From Kaggle

This creates archives in `/kaggle/working/tcrb_kaggle_exports` so you can download them from the Kaggle notebook file browser.

In [14]:
archive_base = EXPORT_ROOT / f"tcrb_export_{RUN_STAMP}"
zip_path = shutil.make_archive(str(archive_base), "zip", root_dir=str(stage_root))
tar_path = shutil.make_archive(str(archive_base), "gztar", root_dir=str(stage_root))

print("Download these from Kaggle:")
print("ZIP:", zip_path)
print("TAR.GZ:", tar_path)
print("Export root contents:")
for path in sorted(EXPORT_ROOT.iterdir()):
    print("-", path)

Download these from Kaggle:
ZIP: /kaggle/working/tcrb_kaggle_exports/tcrb_export_20260424-055301.zip
TAR.GZ: /kaggle/working/tcrb_kaggle_exports/tcrb_export_20260424-055301.tar.gz
Export root contents:
- /kaggle/working/tcrb_kaggle_exports/git_bundle
- /kaggle/working/tcrb_kaggle_exports/staged_artifacts
- /kaggle/working/tcrb_kaggle_exports/tcrb_export_20260424-055301.tar.gz
- /kaggle/working/tcrb_kaggle_exports/tcrb_export_20260424-055301.zip


## 8. Push Changes Or Export Artifacts Back To The Local Git Workflow

The default path is archive export for local download. The optional push path is disabled unless you deliberately turn it on and configure credentials in Kaggle.

In [15]:
patch_script = """
set -euo pipefail
cd /kaggle/working/tool-calling-reliability-benchmark
mkdir -p /kaggle/working/tcrb_kaggle_exports/git_bundle

git status --short --branch | tee /kaggle/working/tcrb_kaggle_exports/git_bundle/git_status.txt
git diff --binary > /kaggle/working/tcrb_kaggle_exports/git_bundle/working_tree.patch || true
git diff --cached --binary > /kaggle/working/tcrb_kaggle_exports/git_bundle/staged.patch || true
git ls-files -m -o --exclude-standard > /kaggle/working/tcrb_kaggle_exports/git_bundle/changed_files.txt || true
"""
subprocess.run(["bash", "-lc", patch_script], check=True, cwd=str(REPO_DIR))
print("Prepared patch exports in", PATCH_EXPORT_DIR)

if PUSH_TO_REMOTE:
    push_script = f"""
set -euo pipefail
cd /kaggle/working/tool-calling-reliability-benchmark
git remote -v
git push {PUSH_REMOTE_NAME} HEAD:{PUSH_BRANCH_NAME}
"""
    print("PUSH_TO_REMOTE enabled. Attempting push.")
    subprocess.run(["bash", "-lc", push_script], check=True, cwd=str(REPO_DIR))
else:
    print("Push disabled. Use the archives and patches for local download and commit.")

## feat/llm...origin/feat/llm
 M uv.lock
Prepared patch exports in /kaggle/working/tcrb_kaggle_exports/git_bundle
Push disabled. Use the archives and patches for local download and commit.
